In [ ]:
# CardioVision AI – ECG Arrhythmia Beat Classifier

**Author:** [khalid azimi]
**Date:** April 2026

## Overview
This notebook implements a **ResNet18‑1D + LoRA** model to classify individual heartbeats from ECG signals into **5 AAMI arrhythmia classes** (N, S, V, F, Q).
- **Dataset:** MIT‑BIH Arrhythmia Database
- **Accuracy:** 91.02% on test set

## Key Features
- **LoRA fine‑tuning** – only 0.43% trainable parameters (memory efficient)
- **Focal Loss + Class Weights** – handles extreme class imbalance
- **Grad‑CAM** – visual explanations for clinical trust




In [ ]:
# @title **DOWNLOAD MIT-BIH ARRHYTHMIA DATASET (Direct)**
import os
import requests
import zipfile
import pandas as pd

# Create directory
os.makedirs("./mitbih_data", exist_ok=True)

# Download preprocessed CSV files (109k+ beats)
urls = {
    "mitbih_train.csv": "https://raw.githubusercontent.com/monajalal/ECG_Heartbeat_Classification/master/data/mitbih_train.csv",
    "mitbih_test.csv": "https://raw.githubusercontent.com/monajalal/ECG_Heartbeat_Classification/master/data/mitbih_test.csv"
}

for fname, url in urls.items():
    print(f"Downloading {fname}...")
    r = requests.get(url, stream=True)
    with open(f"./mitbih_data/{fname}", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
    print(f"Saved {fname}")

# Load and inspect
train = pd.read_csv("./mitbih_data/mitbih_train.csv", header=None)
test = pd.read_csv("./mitbih_data/mitbih_test.csv", header=None)
print(f"\n✅ Data loaded! Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Classes: N(0), S(1), V(2), F(3), Q(4)")
print(f"Sample counts:\n{train.iloc[:, -1].value_counts().sort_index()}")

In [ ]:
# @title **INSTALL WFDB AND DOWNLOAD MIT-BIH ARRHYTHMIA DATABASE**
!pip install -q wfdb

import wfdb
import os
import numpy as np
from tqdm import tqdm

# List of all 48 records in MIT-BIH Arrhythmia Database
record_names = [
    '100','101','102','103','104','105','106','107','108','109',
    '111','112','113','114','115','116','117','118','119','121',
    '122','123','124','200','201','202','203','205','207','208',
    '209','210','212','213','214','215','217','219','220','221',
    '222','223','228','230','231','232','233','234'
]

# Create data directory
os.makedirs('./mitbih_raw', exist_ok=True)

# Download each record (signals and annotations)
for rec in tqdm(record_names, desc="Downloading records"):
    wfdb.dl_database('mitdb', dl_dir='./mitbih_raw', records=[rec], overwrite=False)

print("\n✅ All 48 records downloaded successfully!")

In [ ]:
# @title **SEGMENT BEATS AND CREATE LABELED DATASET**
import wfdb
import numpy as np
from collections import Counter

# AAMI class mapping (N, S, V, F, Q)
aami_map = {
    'N': ['N','L','R','e','j'],          # Normal
    'S': ['A','a','J','S'],              # Supraventricular
    'V': ['V','E'],                      # Ventricular
    'F': ['F'],                          # Fusion
    'Q': ['/','f','Q']                   # Unknown/Paced
}
symbol_to_class = {}
for cls, syms in aami_map.items():
    for s in syms:
        symbol_to_class[s] = cls

def extract_beats(record_name, target_len=187):
    """Extract all annotated beats from a record."""
    sig, fields = wfdb.rdsamp(f'./mitbih_raw/{record_name}')
    ann = wfdb.rdann(f'./mitbih_raw/{record_name}', 'atr')

    # Use first channel (MLII) if available, else first channel
    channel = 0
    if 'MLII' in fields['sig_name']:
        channel = fields['sig_name'].index('MLII')

    beats = []
    labels = []

    for i, sample in enumerate(ann.sample):
        sym = ann.symbol[i]
        if sym not in symbol_to_class:
            continue  # skip unclassified beats

        # Extract window centered at beat
        start = sample - target_len // 2
        end = start + target_len
        if start >= 0 and end <= sig.shape[0]:
            beat = sig[start:end, channel]
            beats.append(beat)
            labels.append(symbol_to_class[sym])

    return beats, labels

# Process all records
all_beats = []
all_labels = []

for rec in tqdm(record_names, desc="Extracting beats"):
    beats, labels = extract_beats(rec)
    all_beats.extend(beats)
    all_labels.extend(labels)

print(f"\nTotal beats extracted: {len(all_beats)}")
print("Class distribution:", Counter(all_labels))

In [ ]:
# @title **PREPARE TRAIN/TEST SPLITS AND NORMALIZE**
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Convert to numpy arrays
X = np.array(all_beats, dtype=np.float32)
y = np.array(all_labels)

# Encode string labels to integers
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = le.classes_
print("Class mapping:", dict(zip(class_names, range(len(class_names)))))

# Train/Test split (80/20) stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Normalize each beat (z-score per beat)
X_train = (X_train - X_train.mean(axis=1, keepdims=True)) / (X_train.std(axis=1, keepdims=True) + 1e-8)
X_test = (X_test - X_test.mean(axis=1, keepdims=True)) / (X_test.std(axis=1, keepdims=True) + 1e-8)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class counts: {np.bincount(y_train)}")
print(f"Test class counts: {np.bincount(y_test)}")

In [ ]:
# @title **CREATE DATALOADERS & OPTIMIZER**
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Convert to tensors
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                             torch.tensor(y_test, dtype=torch.long))

# Batch size reduced for T4 memory safety
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Optimizer (AdamW with weight decay)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler (reduces LR when validation plateaus)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion.to(device)
print(f"Using device: {device}")

In [ ]:
# @title **UPGRADED MODEL: RESNET18-1D + FOCAL LOSS**
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from peft import LoraConfig, get_peft_model

# -------------------------------
# Focal Loss for Imbalanced Data
# -------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha  # class weights tensor
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# -------------------------------
# ResNet18-1D (from scratch)
# -------------------------------
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion*planes:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_planes, self.expansion*planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(self.expansion*planes)
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512*BasicBlock1D.expansion, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(BasicBlock1D(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock1D.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B,1,187)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out).squeeze(-1)
        return self.fc(out)

# -------------------------------
# Build and apply LoRA to FC layer
# -------------------------------
base_model = ResNet18_1D(num_classes=5)
lora_config = LoraConfig(
    r=32,                     # Higher rank for more expressivity
    lora_alpha=64,
    target_modules=["fc"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# -------------------------------
# Loss & Optimizer with Weight Decay
# -------------------------------
class_weights_tensor = torch.tensor([27.28, 0.2416, 2.722, 7.872, 3.026], dtype=torch.float32)
criterion = FocalLoss(alpha=class_weights_tensor, gamma=2.0)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-5)

#  transfer to gpu .
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion.to(device)

In [ ]:
# @title **DATALOADER WITH GRADIENT ACCUMULATION SETUP**
BATCH_SIZE = 32          # Smaller per-step batch  to memory
ACCUM_STEPS = 8          # Simulate batch size = 32 * 8 = 256

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2, shuffle=False)

print(f"Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")

In [ ]:
# @title **FIXED TRAINING LOOP (NO STALE VARIABLES)**

import numpy as np
from tqdm.notebook import tqdm
import time

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

EPOCHS = 20
best_acc = 0.0
model.train()

for epoch in range(EPOCHS):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    optimizer.zero_grad()

    for i, (xb, yb) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)):
        xb, yb = xb.to(device), yb.to(device)

        # Decide whether to use mixup for this batch
        use_mixup = (epoch > 2) and (np.random.rand() < 0.5)

        if use_mixup:
            xb, y_a, y_b, lam = mixup_data(xb, yb, alpha=0.2)
            outputs = model(xb)
            loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            # Accuracy for mixup (soft)
            _, preds = torch.max(outputs, 1)
            correct = (lam * (preds == y_a).float() + (1 - lam) * (preds == y_b).float()).sum().item()
        else:
            outputs = model(xb)
            loss = criterion(outputs, yb)
            _, preds = torch.max(outputs, 1)
            correct = (preds == yb).sum().item()

        loss = loss / ACCUM_STEPS
        loss.backward()

        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUM_STEPS * xb.size(0)
        train_correct += correct
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item() * xb.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_ecg_model_v2.pth")
        print("✅ Best model saved!")

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e} | Time: {epoch_time:.1f}s")

print(f"\n🏆 Training complete! Best validation accuracy: {best_acc:.4f}")

In [ ]:
# @title **FINAL EVALUATION & CLASSIFICATION REPORT**
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load best model
model.load_state_dict(torch.load("/content/drive/MyDrive/best_ecg_model_v2.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = torch.argmax(outputs, 1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

class_names = ['F', 'N', 'Q', 'S', 'V']
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - MIT-BIH Arrhythmia')
plt.show()

In [ ]:
# @title **FIX FOCAL LOSS DEVICE MISMATCH**
# i  should transfer the alpha tensor to gpu
criterion.alpha = criterion.alpha.to(device)
print(f"Alpha device: {criterion.alpha.device}")

In [ ]:
# @title **GRAD-CAM VISUALIZATION FOR ECG BEATS**
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import functional as F

# -------------------------------
# 1. Ensuare model is loaded and in eval mode

model.eval()
print("Model ready for Grad‑CAM.")


# 2. Hook setup for the last convolutional layer

feature_maps = []
gradients = []

def forward_hook(module, input, output):
    feature_maps.append(output)

def backward_hook(module, grad_in, grad_out):
    gradients.append(grad_out[0])

# The last convolutional layer in ResNet18_1D is inside layer4[-1].conv2
target_layer = model.base_model.layer4[-1].conv2
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)
print(f"Hooked to layer: {target_layer}")


# 3. Grad‑CAM function for a single beat

def generate_gradcam(input_tensor, target_class=None):
    """
    input_tensor: (1, 187) normalized ECG beat
    target_class: int (if None, uses predicted class)
    Returns: heatmap (187,) and prediction info
    """
    model.zero_grad()
    input_tensor.requires_grad = True


    output = model(input_tensor)
    pred_class = torch.argmax(output, dim=1).item()

    if target_class is None:
        target_class = pred_class


    class_score = output[0, target_class]
    class_score.backward()

    # Generate heatmap from gradients and feature maps
    pooled_gradients = torch.mean(gradients[0], dim=[0, 2])
    fmap = feature_maps[0]


    for i in range(fmap.shape[1]):
        fmap[:, i, :] *= pooled_gradients[i]

    heatmap = torch.mean(fmap, dim=1).squeeze().cpu().detach().numpy()
    heatmap = np.maximum(heatmap, 0)  # ReLU
    heatmap = heatmap / (np.max(heatmap) + 1e-8)  # Normalize to [0,1]

    # Interpolate to original signal length (187)
    orig_len = 187
    heatmap = np.interp(np.linspace(0, 1, orig_len),
                        np.linspace(0, 1, len(heatmap)),
                        heatmap)

    # Clear stored maps for next call
    feature_maps.clear()
    gradients.clear()

    return heatmap, pred_class, target_class

# -------------------------------
# 4. Visualize Grad‑CAM for a few examples
# -------------------------------
class_names = ['F', 'N', 'Q', 'S', 'V']

def plot_gradcam_example(class_idx, num_examples=1):
    """Plot Grad‑CAM for a random test sample of given class."""
    indices = np.where(y_test == class_idx)[0]
    for i in range(min(num_examples, len(indices))):
        idx = indices[i]
        signal = X_test[idx]
        input_tensor = torch.tensor(signal, dtype=torch.float32).unsqueeze(0).to(device)

        heatmap, pred, target = generate_gradcam(input_tensor, target_class=class_idx)

        plt.figure(figsize=(12, 4))
        plt.subplot(2, 1, 1)
        plt.plot(signal, color='black', linewidth=1.5)
        plt.title(f"True: {class_names[class_idx]} | Predicted: {class_names[pred]} | "
                  f"Confidence: {torch.softmax(model(input_tensor)[0],0)[pred].item():.2%}")
        plt.ylabel("Amplitude")
        plt.grid(True, alpha=0.3)

        plt.subplot(2, 1, 2)
        plt.plot(signal, color='gray', alpha=0.6)
        plt.scatter(range(187), signal, c=heatmap, cmap='jet', s=15, edgecolors='none')
        plt.colorbar(label='Activation Strength')
        plt.ylabel("Amplitude")
        plt.xlabel("Time (sample)")
        plt.title("Grad‑CAM Heatmap (Red = High Influence on Decision)")
        plt.tight_layout()
        plt.show()

# Generate examples for Ventricular (V) and Normal (N)
print("\n--- Ventricular Beat (V) Grad‑CAM ---")
plot_gradcam_example(4, num_examples=2)

print("\n--- Normal Beat (N) Grad‑CAM ---")
plot_gradcam_example(1, num_examples=1)

In [ ]:
# @title **EXPORT MODEL FOR LOCAL USE**
import torch
import torch.nn as nn
import numpy as np
import os
import shutil

# -----------------------------------------------------------------
# 1. Define the model architecture (same as before)
# -----------------------------------------------------------------
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.conv2 = nn.Conv1d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_planes, self.expansion * planes, 1, stride, bias=False),
                nn.BatchNorm1d(self.expansion * planes)
            )
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv1d(1, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512 * BasicBlock1D.expansion, num_classes)
    def _make_layer(self, planes, blocks, stride):
        strides = [stride] + [1] * (blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock1D(self.in_planes, planes, s))
            self.in_planes = planes * BasicBlock1D.expansion
        return nn.Sequential(*layers)
    def forward(self, x):
        # x shape: (batch, 187)
        x = x.unsqueeze(1)  # (batch, 1, 187)
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out).squeeze(-1)
        return self.fc(out)

# -----------------------------------------------------------------
# 2. Load the trained weights into a new model instance
# -----------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18_1D(num_classes=5)
checkpoint = torch.load("/content/drive/MyDrive/best_ecg_model_v2.pth", map_location=device)
model.load_state_dict(checkpoint)  # The saved state_dict is from the LoRA-wrapped model but it actually saved only the base?
# Note: Since we saved model.state_dict() from the LoRA model, it contains both base and lora weights.
# The checkpoint likely includes the 'lora_' keys. We need to load with strict=False if there's a mismatch.
model.load_state_dict(checkpoint, strict=False)
model.eval()
model.to(device)

# -----------------------------------------------------------------
# 3. Save as TorchScript (runs anywhere without class definition)
# -----------------------------------------------------------------
example_input = torch.randn(1, 187).to(device)
traced_model = torch.jit.trace(model, example_input)
traced_model.save("ecg_model.pt")

# Also save class names
import json
class_names = ['F', 'N', 'Q', 'S', 'V']
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("✅ Model exported as 'ecg_model.pt' and 'class_names.json'")